In [9]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

In [2]:
DATA_DIR_A = '/kaggle/input/the-physionet-challenge-2019-dataset/training_setA/training_setA'
all_files = glob.glob(os.path.join(DATA_DIR_A, '*.psv'))


BASE_FEATURES = ['ICULOS', 'HR', 'MAP', 'SBP', 'DBP', 'Resp', 'Temp', 'O2Sat']
HYPOTENSION_THRESHOLD = 65
WINDOW_SIZE = 12
NUM_PATIENTS = 2000

baselines = {'HR': 80, 'MAP': 90, 'SBP': 120, 'DBP': 80, 'Resp': 16, 'Temp': 37.0, 'O2Sat': 98, 'ICULOS': 1}

In [5]:
def process_patient_df(filepath):
    df = pd.read_csv(filepath, sep='|')
    
    if not all(col in df.columns for col in BASE_FEATURES):
        return None
        
    df = df[BASE_FEATURES].copy()
    
    # 🚨 CRITICAL FIX: Sort by actual ICU hour to guarantee chronological order
    df = df.sort_values(by='ICULOS').reset_index(drop=True)
    
    # Handle missing values
    df = df.ffill().bfill().fillna(value=baselines)
    
    # Add Delta Features (Velocity). We skip ICULOS since time velocity is just 1 hour.
    features_to_diff = [c for c in BASE_FEATURES if c != 'ICULOS']
    for col in features_to_diff:
        df[f'{col}_delta'] = df[col].diff().fillna(0)
        
    # Multi-Horizon Targets
    is_hypo = (df['MAP'] < HYPOTENSION_THRESHOLD).astype(int)
 

    df['y_3h'] = (
    is_hypo.rolling(window=3, min_periods=3)
           .max()
           .shift(-2))

    df['y_6h'] = (
        is_hypo.rolling(window=6, min_periods=6)
               .max()
               .shift(-5)
    )
    
    df['y_12h'] = (
        is_hypo.rolling(window=12, min_periods=12)
               .max()
               .shift(-11)
    )
    
    # Drop rows where we cannot look into the future
    df = df.dropna()
    
    return df

In [4]:
sample_df = process_patient_df(all_files[0])
display(sample_df.head(50))

,ICULOS,HR,MAP,SBP,DBP,Resp,Temp,O2Sat,HR_delta,MAP_delta,SBP_delta,DBP_delta,Resp_delta,Temp_delta,O2Sat_delta,y_3h,y_6h,y_12h
0,1,65.0,72.0,129.0,69.0,16.5,35.78,100.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1.0,1.0,1.0
1,2,65.0,72.0,129.0,69.0,16.5,35.78,100.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1.0,1.0,1.0
2,3,78.0,42.5,129.0,69.0,16.5,35.78,100.0,13.0,-29.5,0.0,0.0,0.0,0.00,0.0,1.0,1.0,1.0
3,4,73.0,42.5,129.0,69.0,17.0,35.78,100.0,-5.0,0.0,0.0,0.0,0.5,0.00,0.0,1.0,1.0,1.0
4,5,70.0,74.0,129.0,69.0,14.0,35.78,100.0,-3.0,31.5,0.0,0.0,-3.0,0.00,0.0,0.0,0.0,0.0
5,6,62.0,85.0,124.0,61.0,14.0,35.78,100.0,-8.0,11.0,-5.0,-8.0,0.0,0.00,0.0,0.0,0.0,0.0
6,7,61.0,75.0,101.0,58.0,14.0,35.78,100.0,-1.0,-10.0,-23.0,-3.0,0.0,0.00,0.0,0.0,0.0,0.0
7,8,68.0,93.5,142.0,78.0,16.0,35.78,100.0,7.0,18.5,41.0,20.0,2.0,0.00,0.0,0.0,0.0,0.0
8,9,71.0,74.0,121.0,91.0,14.0,35.78,100.0,3.0,-19.5,-21.0,13.0,-2.0,0.00,0.0,0.0,0.0,0.0
9,10,69.0,79.0,120.0,98.0,14.0,35.78,100.0,-2.0,5.0,-1.0,7.0,0.0,0.00,0.0,0.0,0.0,0.0


In [6]:
X_windows, y_windows = [], []

for filepath in tqdm(all_files[:NUM_PATIENTS]):
    df = process_patient_df(filepath)
    
    if df is None or len(df) < WINDOW_SIZE:
        continue
   
    target_cols = ['y_3h', 'y_6h', 'y_12h']
    feature_cols = [c for c in df.columns if c not in target_cols]
    

    for i in range(len(df) - WINDOW_SIZE + 1):
        window_X = df[feature_cols].iloc[i : i + WINDOW_SIZE].values
        window_y = df[target_cols].iloc[i + WINDOW_SIZE - 1].values
        
        X_windows.append(window_X)
        y_windows.append(window_y)

X_final = np.array(X_windows)
y_final = np.array(y_windows)

y_final[:50, :]

100%|██████████| 2000/2000 [01:12<00:00, 27.53it/s]


array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 1., 1.],
       [0., 1., 1.],
       [0., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 1., 1.],
       [0., 1., 1.],
       [0., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0

In [7]:

print(f"Input Shape (Batch, Time, Features): {X_final.shape}")
print(f"Target Shape (Batch, Horizons): {y_final.shape}")

Input Shape (Batch, Time, Features): (36231, 12, 15)
Target Shape (Batch, Horizons): (36231, 3)


In [8]:
train_files, temp_files = train_test_split(all_files[:NUM_PATIENTS], test_size=0.3, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

print(f"Patient Split -> Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

Patient Split -> Train: 1400 | Val: 300 | Test: 300


In [11]:
def extract_windows_from_files(file_list):
    X_list, y_list = [], []
    for filepath in tqdm(file_list, desc="Processing files"):
        df = process_patient_df(filepath)
        if df is None or len(df) < WINDOW_SIZE:
            continue
            
        target_cols = ['y_3h', 'y_6h', 'y_12h']
        feature_cols = [c for c in df.columns if c not in target_cols]
        
        for i in range(len(df) - WINDOW_SIZE + 1):
            X_list.append(df[feature_cols].iloc[i : i + WINDOW_SIZE].values)
            y_list.append(df[target_cols].iloc[i + WINDOW_SIZE - 1].values)
            
    return np.array(X_list), np.array(y_list)


In [12]:
X_train_raw, y_train = extract_windows_from_files(train_files)


Processing files: 100%|██████████| 1400/1400 [00:48<00:00, 28.93it/s]


In [17]:
X_val_raw, y_val = extract_windows_from_files(val_files)

Processing files: 100%|██████████| 300/300 [00:09<00:00, 30.77it/s]


In [18]:

X_test_raw, y_test = extract_windows_from_files(test_files)

Processing files: 100%|██████████| 300/300 [00:09<00:00, 31.51it/s]


In [20]:
scaler = StandardScaler()
batch_train, time_steps, num_features = X_train_raw.shape
X_train_2d = X_train_raw.reshape(-1, num_features)


scaler.fit(X_train_2d)

scaler.fit(X_train_2d)

X_train = scaler.transform(X_train_2d).reshape(batch_train, time_steps, num_features)
X_val = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(X_val_raw.shape[0], time_steps, num_features)
X_test = scaler.transform(X_test_raw.reshape(-1, num_features)).reshape(X_test_raw.shape[0], time_steps, num_features)


In [21]:
pos_weights = []
horizons = ['3h', '6h', '12h']

for i in range(3):
    pos_count = np.sum(y_train[:, i] == 1)
    neg_count = np.sum(y_train[:, i] == 0)
    
    weight = neg_count / (pos_count + 1e-5) 
    pos_weights.append(weight)
    
    print(f"Horizon {horizons[i]} -> Stable: {neg_count} | Crashes: {pos_count} | Recommended Weight: {weight:.2f}")

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)

print(f"Final Train Shape: {X_train.shape}")

Horizon 3h -> Stable: 19321 | Crashes: 6920 | Recommended Weight: 2.79
Horizon 6h -> Stable: 16868 | Crashes: 9373 | Recommended Weight: 1.80
Horizon 12h -> Stable: 14013 | Crashes: 12228 | Recommended Weight: 1.15
Final Train Shape: (26241, 12, 15)


In [22]:
print(f"pos_weight_tensor: {pos_weight_tensor}")

pos_weight_tensor: tensor([2.7921, 1.7996, 1.1460])


In [23]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
import numpy as np

In [24]:
BATCH_SIZE = 32

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [25]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super(CausalConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        if self.padding > 0: x = x[:, :, :-self.padding]
        return self.relu(x)

class FedSepsisModel(nn.Module):
    def __init__(self, input_features=15, embed_dim=64, num_heads=4):
        super(FedSepsisModel, self).__init__()
        self.embedding = nn.Linear(input_features, embed_dim)
        
        self.tcn = nn.Sequential(
            CausalConv1d(embed_dim, embed_dim, kernel_size=3, dilation=1),
            CausalConv1d(embed_dim, embed_dim, kernel_size=3, dilation=2),
            CausalConv1d(embed_dim, embed_dim, kernel_size=3, dilation=4)
        )
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        
        self.head_3h = nn.Sequential(nn.Linear(embed_dim, 32), nn.ReLU(), nn.Linear(32, 1))
        self.head_6h = nn.Sequential(nn.Linear(embed_dim, 32), nn.ReLU(), nn.Linear(32, 1))
        self.head_12h = nn.Sequential(nn.Linear(embed_dim, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.tcn(x)
        x = x.permute(0, 2, 1)
        attn_out, attn_weights = self.attention(x, x, x)
        patient_state = attn_out[:, -1, :] 
        
        out_3h = self.head_3h(patient_state)
        out_6h = self.head_6h(patient_state)
        out_12h = self.head_12h(patient_state)
        
        return torch.cat([out_3h, out_6h, out_12h], dim=1), attn_weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Training on: {device}")

model = FedSepsisModel().to(device)

pos_weight_tensor = pos_weight_tensor.to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10 # Keep it low for fast hackathon iteration

for epoch in range(EPOCHS):
    # -- TRAIN PASS --
    model.train()
    train_loss = 0.0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        logits, _ = model(inputs)
        
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # -- VALIDATION PASS --
    model.eval()
    val_loss = 0.0
    all_targets = []
    all_preds = []
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits, _ = model(inputs)
            loss = criterion(logits, targets)
            val_loss += loss.item()
            
            # Apply Sigmoid here just for the AUROC metric calculation
            preds = torch.sigmoid(logits)
            
            all_targets.append(targets.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    
    # Calculate AUROC for each horizon
    all_targets = np.vstack(all_targets)
    all_preds = np.vstack(all_preds)
    
    try:
        auroc_3h = roc_auc_score(all_targets[:, 0], all_preds[:, 0])
        auroc_6h = roc_auc_score(all_targets[:, 1], all_preds[:, 1])
        auroc_12h = roc_auc_score(all_targets[:, 2], all_preds[:, 2])
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"          AUROC -> 3h: {auroc_3h:.3f} | 6h: {auroc_6h:.3f} | 12h: {auroc_12h:.3f}")
    except ValueError:
        # Failsafe just in case a batch has only 1 class during the first few steps
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

print("\n🎉 MODEL TRAINING COMPLETE!")

🚀 Training on: cuda
Epoch [1/10] | Train Loss: 0.5493 | Val Loss: 0.4915
          AUROC -> 3h: 0.932 | 6h: 0.912 | 12h: 0.890
Epoch [2/10] | Train Loss: 0.5017 | Val Loss: 0.4786
          AUROC -> 3h: 0.932 | 6h: 0.910 | 12h: 0.889
Epoch [3/10] | Train Loss: 0.4930 | Val Loss: 0.4805
          AUROC -> 3h: 0.933 | 6h: 0.911 | 12h: 0.888
Epoch [4/10] | Train Loss: 0.4797 | Val Loss: 0.4861
          AUROC -> 3h: 0.931 | 6h: 0.909 | 12h: 0.888
Epoch [5/10] | Train Loss: 0.4682 | Val Loss: 0.4903
          AUROC -> 3h: 0.930 | 6h: 0.907 | 12h: 0.885
Epoch [6/10] | Train Loss: 0.4510 | Val Loss: 0.4983
          AUROC -> 3h: 0.932 | 6h: 0.909 | 12h: 0.887
Epoch [7/10] | Train Loss: 0.4332 | Val Loss: 0.5048
          AUROC -> 3h: 0.929 | 6h: 0.909 | 12h: 0.885
Epoch [8/10] | Train Loss: 0.4139 | Val Loss: 0.5327
          AUROC -> 3h: 0.923 | 6h: 0.897 | 12h: 0.869
Epoch [9/10] | Train Loss: 0.3925 | Val Loss: 0.5555
          AUROC -> 3h: 0.920 | 6h: 0.896 | 12h: 0.871
Epoch [10/10] | T

In [27]:
import torch
import torch.nn as nn
import copy
from sklearn.metrics import roc_auc_score
import numpy as np

class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.5):
        super(CausalConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.conv(x)
        if self.padding > 0: x = x[:, :, :-self.padding]
        return self.dropout(self.relu(x))

# 🛡️ HEAVY ARMOR: embed_dim=32, dropout=0.5
class FedSepsisModel(nn.Module):
    def __init__(self, input_features=15, embed_dim=32, num_heads=4, dropout=0.5):
        super(FedSepsisModel, self).__init__()
        self.embedding = nn.Linear(input_features, embed_dim)
        
        self.tcn = nn.Sequential(
            CausalConv1d(embed_dim, embed_dim, kernel_size=3, dilation=1, dropout=dropout),
            CausalConv1d(embed_dim, embed_dim, kernel_size=3, dilation=2, dropout=dropout),
            CausalConv1d(embed_dim, embed_dim, kernel_size=3, dilation=4, dropout=dropout)
        )
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True, dropout=dropout)
        
        self.head_3h = nn.Sequential(nn.Linear(embed_dim, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1))
        self.head_6h = nn.Sequential(nn.Linear(embed_dim, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1))
        self.head_12h = nn.Sequential(nn.Linear(embed_dim, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1))

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.tcn(x)
        x = x.permute(0, 2, 1)
        attn_out, attn_weights = self.attention(x, x, x)
        patient_state = attn_out[:, -1, :] 
        
        out_3h = self.head_3h(patient_state)
        out_6h = self.head_6h(patient_state)
        out_12h = self.head_12h(patient_state)
        
        return torch.cat([out_3h, out_6h, out_12h], dim=1), attn_weights

# --- TRAINING LOOP SETUP ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Re-initialize the shrunk model
model = FedSepsisModel().to(device)

pos_weight_tensor = pos_weight_tensor.to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# 🛡️ HEAVY ARMOR: weight_decay=1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

EPOCHS = 15
patience = 4 # Giving it slightly more patience since extreme dropout makes training noisier
best_val_loss = float('inf')
best_model_weights = None
epochs_no_improve = 0

print("🚀 Starting HEAVY ARMOR Training (Lower Capacity, 50% Dropout, 10x Weight Decay)...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits, _ = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    all_targets, all_preds = [], []
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits, _ = model(inputs)
            loss = criterion(logits, targets)
            val_loss += loss.item()
            
            preds = torch.sigmoid(logits)
            all_targets.append(targets.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    all_targets = np.vstack(all_targets)
    all_preds = np.vstack(all_preds)
    
    try:
        auroc_3h = roc_auc_score(all_targets[:, 0], all_preds[:, 0])
        auroc_6h = roc_auc_score(all_targets[:, 1], all_preds[:, 1])
        auroc_12h = roc_auc_score(all_targets[:, 2], all_preds[:, 2])
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"          AUROC -> 3h: {auroc_3h:.3f} | 6h: {auroc_6h:.3f} | 12h: {auroc_12h:.3f}")
    except ValueError:
        pass

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\n🛑 Early stopping triggered at Epoch {epoch+1}! Restoring best weights.")
            break

model.load_state_dict(best_model_weights)
print("\n🎉 MODEL SECURED.")

🚀 Starting HEAVY ARMOR Training (Lower Capacity, 50% Dropout, 10x Weight Decay)...
Epoch [1/15] | Train Loss: 0.6553 | Val Loss: 0.5125
          AUROC -> 3h: 0.921 | 6h: 0.905 | 12h: 0.882
Epoch [2/15] | Train Loss: 0.5823 | Val Loss: 0.4886
          AUROC -> 3h: 0.930 | 6h: 0.907 | 12h: 0.885
Epoch [3/15] | Train Loss: 0.5753 | Val Loss: 0.4918
          AUROC -> 3h: 0.929 | 6h: 0.908 | 12h: 0.885
Epoch [4/15] | Train Loss: 0.5664 | Val Loss: 0.5437
          AUROC -> 3h: 0.910 | 6h: 0.897 | 12h: 0.878
Epoch [5/15] | Train Loss: 0.5650 | Val Loss: 0.5007
          AUROC -> 3h: 0.922 | 6h: 0.902 | 12h: 0.879
Epoch [6/15] | Train Loss: 0.5588 | Val Loss: 0.4799
          AUROC -> 3h: 0.931 | 6h: 0.908 | 12h: 0.884
Epoch [7/15] | Train Loss: 0.5616 | Val Loss: 0.4797
          AUROC -> 3h: 0.931 | 6h: 0.909 | 12h: 0.886
Epoch [8/15] | Train Loss: 0.5582 | Val Loss: 0.4813
          AUROC -> 3h: 0.931 | 6h: 0.910 | 12h: 0.886
Epoch [9/15] | Train Loss: 0.5536 | Val Loss: 0.4784
        

In [28]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. PREDICT ON VALIDATION SET ---
model.eval()
all_targets_sim = []
all_preds_sim = []

print("Running clinical simulation on validation patients...")
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        logits, _ = model(inputs)
        preds = torch.sigmoid(logits) # Convert raw logits to probabilities (0 to 1)
        
        all_targets_sim.append(targets.cpu().numpy())
        all_preds_sim.append(preds.cpu().numpy())

all_targets_sim = np.vstack(all_targets_sim)
all_preds_sim = np.vstack(all_preds_sim)

# We will analyze the 3-hour prediction horizon (Index 0)
y_true_3h = all_targets_sim[:, 0]
y_prob_3h = all_preds_sim[:, 0]

# --- 2. CLINICAL METRICS EVALUATION ---
# If our model is > 50% confident, we sound the early warning alarm
MODEL_THRESHOLD = 0.50 
y_pred_binary = (y_prob_3h > MODEL_THRESHOLD).astype(int)

# Calculate Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y_true_3h, y_pred_binary).ravel()

total_windows = len(y_true_3h)
actual_crashes = tp + fn
predicted_alarms = tp + fp

sensitivity = recall_score(y_true_3h, y_pred_binary)
precision = precision_score(y_true_3h, y_pred_binary)

print("\n" + "="*50)
print("🏥 ALARM FATIGUE & EFFICACY SIMULATION (3-Hour Horizon)")
print("="*50)
print(f"Total 12-Hour Patient Windows Evaluated: {total_windows:,}")
print(f"Actual Clinical Crashes (MAP < 65):        {actual_crashes:,}")
print("-" * 50)
print(f"🟢 True Positives (Crashes caught early):  {tp:,} (Sensitivity: {sensitivity*100:.1f}%)")
print(f"🔴 False Positives (False alarms):         {fp:,}")
print(f"🟡 False Negatives (Missed crashes):       {fn:,}")
print(f"✅ True Negatives (Correctly kept quiet):  {tn:,}")
print("-" * 50)

# Calculate the reduction in unnecessary noise
# A traditional monitor alarms constantly. Our model's precision tells us 
# how many of our alarms were actually meaningful.
print(f"🚨 Alarm Precision: When our model beeps, it is correct {precision*100:.1f}% of the time.")
print("="*50)

Running clinical simulation on validation patients...

🏥 ALARM FATIGUE & EFFICACY SIMULATION (3-Hour Horizon)
Total 12-Hour Patient Windows Evaluated: 5,104
Actual Clinical Crashes (MAP < 65):        1,442
--------------------------------------------------
🟢 True Positives (Crashes caught early):  1,196 (Sensitivity: 82.9%)
🔴 False Positives (False alarms):         504
🟡 False Negatives (Missed crashes):       246
✅ True Negatives (Correctly kept quiet):  3,158
--------------------------------------------------
🚨 Alarm Precision: When our model beeps, it is correct 70.4% of the time.
